In [ ]:
#from proto import *
#from engine import *
#from utils import *
#from runners import CompartmentalModel

# from experimental import *
#from managed import *

#from categories import *

import pandas as pd
import numpy as np
import datetime as dt

pd.options.plotting.backend = "plotly"

#import computegraph as cg

from graph import *


In [ ]:
from epi import *

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

In [ ]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))
loc_strat = humans.stratify(Stratification("location", ["N", "S", "E", "W"]))

In [ ]:
mm_data = np.array([
        [2.0, 1.0],
        [0.0, 0.0]
    ])

In [ ]:
age_cats = age_strat.categories()
infectees = age_cats
infectors = age_cats

mm = mixing_matrix(mm_data, infectors, infectees)
mm.indices

In [ ]:
iprocess = defer(InfectionProcess)(mm, infectees, infectors, disease_state["I"])

In [ ]:
foi = defer(InfectionProcess.process)(
    iprocess, CompartmentValues, Parameter("contact_rate", 0.2)
)

In [ ]:
infection = TransitionFlow("infection", disease_state["S"], disease_state["I"], foi)
recovery = TransitionFlow("recovery",
    disease_state["I"], disease_state["R"], Parameter("recovery_rate", 0.1)
)

In [ ]:
infection.adjustments.append(CategoryData(loc_strat.categories(), np.array([0.0, 1.0, 0.5, 1.1])))

In [ ]:
times = pd.date_range("7 jun 1980", "7 december 1980")
epi_model = CompartmentalEpiModel(humans, times)

epi_model.add_flow(infection)
epi_model.add_flow(recovery)

In [ ]:
pop_data = pd.Series(index=["N","E","S","W"], 
                     data=np.array([1000.0,1500.0,200.0,500.0]))

base_pops = strat_data_from_pandas(pop_data, loc_strat)
#base_pops = cat_data_from_pandas(df, loc_strat.categories().product(age_strat.categories()),"pop")
pop_splits = [
    CategoryData(disease_state.categories(), jnp.array(([0.9,0.1,0.0])))
]

epi_model.set_initial_population(base_pops, pop_splits)


In [ ]:
params = {"contact_rate": 0.2, "recovery_rate": 0.01}

results = epi_model.run(params)

In [ ]:
compres = results["compartments"]
flowres = results["flows"]

In [ ]:
compres

In [ ]:
compres.sumcats(compartment=loc_strat.categories()).to_pandas_df().plot()

In [ ]:
#type: ignore
non_infectious = Category(disease_state["S","R"]) 
infectious = Category(disease_state["I"])
inf_status_groups = CategoryGroup([non_infectious,infectious]).product(age_strat.categories())

compres.sumcats(compartment=inf_status_groups).to_pandas_df().plot()

In [ ]:
infres = flowres["infection"]
infres

In [ ]:
infres.to_pandas_df().plot()

In [ ]:
# Zoom in and show only age cats
infres.query(time=np.s_[:"aug 1 1980"]).sumcats(dest=age_cats).to_pandas_df().plot()

In [ ]:
mm

In [ ]:
mm.query(dest=age_strat["adult"])

In [ ]:
results["compartments"].sumcats(compartment=disease_state.categories()).to_pandas_df().plot()